In [1]:
import requests
import pandas as pd

def fetch_temperature(name, longitude, latitude, start_date, end_date):

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m",
        "timezone": "Pacific/Auckland"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame({
        "time":         data["hourly"]["time"],
        "temperature":  data["hourly"]["temperature_2m"]
    })

    df["time"] = pd.to_datetime(df["time"])
    print(df.head(10))
    print(f"\nShape: {df.shape}")
    print(f"Missing values: {df.isnull().sum().sum()}")

    df = df.rename(columns={"temperature": f"{name}_temp_c"})
    return df

In [2]:
#             name,           longitude,   latitude,  start_date,   end_date
request =  [['Auckland'     , 174.7633, -36.8485, "2014-01-01", "2026-05-25"],
            ['Christchurch' , 172.6362, -43.5321, "2014-01-01", "2026-05-25"],
            ['Wellington'   , 174.7762, -41.2865, "2014-01-01", "2026-05-25"],
            ['Hamilton'     , 175.2793, -37.7870, "2014-01-01", "2026-05-25"],
            ['Tauranga'     , 176.1651, -37.6878, "2014-01-01", "2026-05-25"],
            ['Dunedin'      , 170.5028, -45.8788, "2014-01-01", "2026-05-25"]]

temp_data = pd.DataFrame()
for name, lon, lat, start, end in request:
    df = fetch_temperature(name, lon, lat, start, end)

    if temp_data.empty:
        temp_data = df
    else:
        temp_data = temp_data.merge(df, on="time", how="outer")

print(temp_data.head(5))

                 time  temperature
0 2014-01-01 00:00:00         16.6
1 2014-01-01 01:00:00         15.9
2 2014-01-01 02:00:00         14.6
3 2014-01-01 03:00:00         14.5
4 2014-01-01 04:00:00         13.7
5 2014-01-01 05:00:00         13.5
6 2014-01-01 06:00:00         14.0
7 2014-01-01 07:00:00         17.3
8 2014-01-01 08:00:00         18.6
9 2014-01-01 09:00:00         19.5

Shape: (108672, 2)
Missing values: 0
                 time  temperature
0 2014-01-01 00:00:00         11.9
1 2014-01-01 01:00:00         11.2
2 2014-01-01 02:00:00         10.5
3 2014-01-01 03:00:00         10.0
4 2014-01-01 04:00:00          9.9
5 2014-01-01 05:00:00         10.2
6 2014-01-01 06:00:00         11.3
7 2014-01-01 07:00:00         13.8
8 2014-01-01 08:00:00         14.7
9 2014-01-01 09:00:00         15.2

Shape: (108672, 2)
Missing values: 0
                 time  temperature
0 2014-01-01 00:00:00         15.8
1 2014-01-01 01:00:00         15.8
2 2014-01-01 02:00:00         15.8
3 2014-01-01 0

In [3]:
temp_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108672 entries, 0 to 108671
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   time                 108672 non-null  datetime64[ns]
 1   Auckland_temp_c      108672 non-null  float64       
 2   Christchurch_temp_c  108672 non-null  float64       
 3   Wellington_temp_c    108672 non-null  float64       
 4   Hamilton_temp_c      108672 non-null  float64       
 5   Tauranga_temp_c      108672 non-null  float64       
 6   Dunedin_temp_c       108672 non-null  float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 5.8 MB


In [4]:
temp_data.to_csv("Temperature_data.csv", index=False)